In [2]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
musk = pd.read_csv("musk1.csv", low_memory=False)
musk.head()

,Molecule_name,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F158,F159,F160,F161,F162,F163,F164,F165,F166,Class
0,MUSK-jf59,47.0,-110.0,-28.0,31.0,-117.0,-84.0,52.0,-2.0,24.0,...,-209.0,-206.0,-13.0,0.0,29.0,45.0,-44.0,-5.0,-8.0,1
1,MUSK-jf59,9.0,-114.0,-21.0,26.0,-117.0,-13.0,-166.0,47.0,-221.0,...,-51.0,24.0,-30.0,-58.0,-17.0,119.0,-28.0,-28.0,112.0,1
2,MUSK-jf59,49.0,-98.0,-26.0,29.0,-117.0,-83.0,66.0,-10.0,44.0,...,-212.0,-210.0,-13.0,5.0,21.0,48.0,-45.0,-13.0,-12.0,1
3,MUSK-jf59,52.0,-110.0,-21.0,25.0,-117.0,61.0,-161.0,73.0,-218.0,...,14.0,-26.0,-30.0,12.0,8.0,85.0,-52.0,-60.0,-29.0,1
4,MUSK-jf59,23.0,-113.0,-21.0,35.0,-117.0,80.0,-161.0,66.0,-216.0,...,34.0,36.0,-31.0,-60.0,2.0,96.0,-31.0,-9.0,90.0,1


In [4]:
musk[' Class'].unique()

array([1, 0])

Postoji problem u nazivu varijable, ne mogu točno odrediti koji pa ću koristiti columns.

In [5]:
display(musk.describe())
display("Unikatne vrijednosti Class stupca", musk[musk.columns[-1]].unique())

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F158,F159,F160,F161,F162,F163,F164,F165,F166,Class
count,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,...,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000,476.000000
mean,38.731092,-120.142857,-79.243697,15.903361,-112.298319,-9.760504,-16.628151,7.762605,-91.048319,-76.077731,...,-204.405462,-108.262605,-25.231092,37.428571,79.752101,85.363445,-34.128151,-26.241597,33.714286,0.434874
std,18.087948,88.313464,69.172519,75.515959,28.704947,92.025496,106.560891,75.786461,96.321303,72.467693,...,89.566426,121.794583,65.588295,87.315888,49.269244,47.487575,16.019277,58.009205,54.189087,0.496262
min,-9.000000,-199.000000,-166.000000,-115.000000,-117.000000,-184.000000,-170.000000,-231.000000,-242.000000,-284.000000,...,-324.000000,-217.000000,-135.000000,-126.000000,-78.000000,35.000000,-132.000000,-258.000000,-72.000000,0.000000
25%,31.000000,-193.000000,-144.000000,-53.000000,-117.000000,-89.000000,-159.000000,-9.250000,-215.000000,-118.250000,...,-264.000000,-208.000000,-91.000000,-39.250000,33.000000,54.000000,-42.250000,-28.000000,-12.000000,0.000000
50%,42.000000,-144.500000,-108.000000,28.000000,-117.000000,11.000000,41.000000,18.000000,-41.000000,-73.500000,...,-236.000000,-189.500000,-15.000000,31.500000,84.000000,69.000000,-36.000000,-11.000000,35.000000,0.000000
75%,50.000000,-101.000000,-21.000000,38.000000,-117.000000,70.250000,51.000000,57.000000,-30.000000,-28.000000,...,-154.750000,30.000000,22.000000,128.000000,119.000000,99.000000,-28.000000,7.000000,74.000000,1.000000
max,130.000000,98.000000,83.000000,157.000000,238.000000,200.000000,214.000000,188.000000,135.000000,218.000000,...,72.000000,173.000000,185.000000,253.000000,291.000000,302.000000,24.000000,82.000000,235.000000,1.000000


'Unikatne vrijednosti Class stupca'

array([1, 0])

Prema opisu dataseta i analizom dataseta možemo zaključiti da je target varijabla "Class" koja govori da li se molekula lijeka uspjela vezati na protein. Provjeravamo dataset za za prazne vrijednosti.

In [6]:
musk.isnull().values.any()

np.False_

U datasetu ne postoje vrijednosti koje nedostaju.

In [7]:
target = musk[musk.columns[-1]]
features_num = musk.select_dtypes(include=[np.number])
features = features_num.drop(columns=musk.columns[-1]).select_dtypes(include=[np.number])
features.head()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F157,F158,F159,F160,F161,F162,F163,F164,F165,F166
0,47.0,-110.0,-28.0,31.0,-117.0,-84.0,52.0,-2.0,24.0,-60.0,...,-230.0,-209.0,-206.0,-13.0,0.0,29.0,45.0,-44.0,-5.0,-8.0
1,9.0,-114.0,-21.0,26.0,-117.0,-13.0,-166.0,47.0,-221.0,-48.0,...,-238.0,-51.0,24.0,-30.0,-58.0,-17.0,119.0,-28.0,-28.0,112.0
2,49.0,-98.0,-26.0,29.0,-117.0,-83.0,66.0,-10.0,44.0,-114.0,...,-236.0,-212.0,-210.0,-13.0,5.0,21.0,48.0,-45.0,-13.0,-12.0
3,52.0,-110.0,-21.0,25.0,-117.0,61.0,-161.0,73.0,-218.0,28.0,...,-240.0,14.0,-26.0,-30.0,12.0,8.0,85.0,-52.0,-60.0,-29.0
4,23.0,-113.0,-21.0,35.0,-117.0,80.0,-161.0,66.0,-216.0,28.0,...,-214.0,34.0,36.0,-31.0,-60.0,2.0,96.0,-31.0,-9.0,90.0


In [8]:
train_features, test_features, train_target, test_target = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)
svc = SVC(kernel='linear')
cv_score = cross_val_score(svc, features, target, cv=10, scoring='accuracy')
svc.fit(train_features, train_target)
predictions = svc.predict(test_features)

In [9]:
cm = confusion_matrix(test_target, predictions)
cr = classification_report(test_target, predictions)

In [10]:
print('Confusion matrix\n', cm)
print('Classification report\n', cr)

Confusion matrix
 [[47  7]
 [ 5 37]]
Classification report
               precision    recall  f1-score   support

           0       0.90      0.87      0.89        54
           1       0.84      0.88      0.86        42

    accuracy                           0.88        96
   macro avg       0.87      0.88      0.87        96
weighted avg       0.88      0.88      0.88        96

